In [0]:
from pyspark.sql import functions as F

In [0]:
volume_base_path = '/Volumes/de/p1/raw_volume'

print('Starting to ingest data into Bronze layer ...')

Starting to ingest data into Bronze layer ...


In [0]:
df_users_raw = spark.read.parquet(f'{volume_base_path}/users/users.parquet')
df_users_bronze = df_users_raw \
    .withColumn('ingestion_timestamp', F.current_timestamp()) \
    .withColumn('source_system', F.lit('crm_users'))

df_users_bronze.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('de.p1.bronze_users')

In [0]:
df_products_raw = spark.read.parquet(f'{volume_base_path}/products/products.parquet')
df_products_bronze = df_products_raw \
    .withColumn('ingestion_timestamp', F.current_timestamp()) \
    .withColumn('source_system', F.lit('erp_products'))

df_products_bronze.write \
    .format('delta') \
    .saveAsTable('de.p1.bronze_products')

In [0]:
df_orders_raw = spark.read.parquet(f'{volume_base_path}/orders/*/*.parquet')
df_orders_bronze = df_orders_raw \
    .withColumn('ingestion_timestamp', F.current_timestamp()) \
    .withColumn('source_file_date', F.col('order_timestamp').substr(1, 10))

df_orders_bronze.write \
    .format('delta') \
    .saveAsTable('de.p1.bronze_orders')

In [0]:
df_events_raw = spark.read.json(f'{volume_base_path}/events/clickstream.json')
df_events_bronze = df_events_raw \
    .withColumn('ingestion_timestamp', F.current_timestamp())

df_events_bronze.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('de.p1.bronze_events')

print('All 4 tables in bronze have been successfully created')

All 4 tables in bronze have been successfully created


In [0]:
display(spark.sql('select "Users" as tbl, count(*) as cnt from de.p1.bronze_users union all '
                  'select "Products", count(*) from de.p1.bronze_products union all '
                  'select "Orders", count(*) from de.p1.bronze_orders union all '
                  'select "Events", count(*) from de.p1.bronze_events'))

tbl,cnt
Users,100
Products,50
Orders,600
Events,500
